# Notebook 05 — Cohort & Retention Analysis

## Objetivo
Analisar a evolução das **coortes de aquisição** do e-commerce Olist, medindo retenção mensal, recompra acumulada, receita por safra e diferenças de comportamento associadas ao meio de pagamento da primeira compra.

Este notebook complementa o Notebook 04. A análise de recompra em janelas fixas (30/60/90/180 dias) responde **se o cliente voltou dentro de uma janela**; a análise de coorte responde **como cada safra de aquisição evolui ao longo dos meses**.

### Perguntas de negócio
- Quantos clientes são adquiridos em cada mês?
- Qual percentual volta em M1, M2, M3, M6 e M12?
- A retenção melhorou ou piorou entre safras de aquisição?
- Quanto da receita inicial de uma coorte reaparece nos meses seguintes?
- Qual o valor acumulado por cliente adquirido ao longo do tempo?
- Clientes adquiridos com diferentes meios de pagamento apresentam padrões distintos de retenção?
- Quais coortes merecem investigação por desempenho acima ou abaixo da média?

> **Importante:** em e-commerce transacional, retenção mensal tende a ser muito menor do que em negócios de assinatura. Por isso, além da retenção mensal exata, também será calculada a **recompra acumulada**.

## 1. Configuração e carregamento

O notebook prefere `data/processed/orders_base.csv`, criado no Notebook 01. Caso ele não esteja disponível, reconstrói uma base mínima de pedidos usando os arquivos raw. Essa estratégia mantém o notebook reproduzível sem alterar a granularidade: **uma linha por pedido**.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

PROJECT_ROOT = Path('/mnt/data/olist_project') if Path('/mnt/data/olist_project').exists() else Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
FALLBACK_RAW_DIR = Path('/mnt/data/olist_data')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def raw_path(filename):
    p = RAW_DIR / filename
    if p.exists():
        return p
    fallback = FALLBACK_RAW_DIR / filename
    if fallback.exists():
        return fallback
    raise FileNotFoundError(f'Arquivo não encontrado: {filename}')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)

In [ ]:
orders_base_path = PROCESSED_DIR / 'orders_base.csv'

if orders_base_path.exists():
    orders_base = pd.read_csv(
        orders_base_path,
        parse_dates=['order_purchase_timestamp']
    )
    print('orders_base carregado do Notebook 01:', orders_base.shape)
else:
    print('orders_base não encontrado; reconstruindo base mínima a partir dos arquivos raw...')
    orders = pd.read_csv(
        raw_path('olist_orders_dataset.csv'),
        parse_dates=['order_purchase_timestamp']
    )
    customers = pd.read_csv(raw_path('olist_customers_dataset.csv'))
    payments_raw = pd.read_csv(raw_path('olist_order_payments_dataset.csv'))
    items_raw = pd.read_csv(raw_path('olist_order_items_dataset.csv'))

    payment_order = (
        payments_raw.groupby('order_id', as_index=False)
        .agg(payment_value=('payment_value', 'sum'))
    )
    item_order = (
        items_raw.groupby('order_id', as_index=False)
        .agg(
            item_count=('order_item_id', 'count'),
            product_value=('price', 'sum'),
            freight_value=('freight_value', 'sum')
        )
    )

    orders_base = (
        orders
        .merge(
            customers[['customer_id', 'customer_unique_id']],
            on='customer_id', how='left', validate='one_to_one'
        )
        .merge(payment_order, on='order_id', how='left', validate='one_to_one')
        .merge(item_order, on='order_id', how='left', validate='one_to_one')
    )
    orders_base.to_csv(orders_base_path, index=False)
    print('orders_base reconstruído e exportado:', orders_base.shape)

payments = pd.read_csv(raw_path('olist_order_payments_dataset.csv'))
print('payments:', payments.shape)

## 2. Universo analítico

Mantemos somente pedidos:
- entregues (`order_status == delivered`);
- com `customer_unique_id` válido;
- com data de compra válida.

A data de referência é `order_purchase_timestamp`. O mês de compra será representado como período mensal (`YYYY-MM`).

In [ ]:
delivered = orders_base.loc[
    (orders_base['order_status'] == 'delivered') &
    orders_base['customer_unique_id'].notna() &
    orders_base['order_purchase_timestamp'].notna()
].copy()

delivered['order_purchase_timestamp'] = pd.to_datetime(delivered['order_purchase_timestamp'])
delivered = delivered.sort_values(['customer_unique_id', 'order_purchase_timestamp', 'order_id']).reset_index(drop=True)
delivered['order_month'] = delivered['order_purchase_timestamp'].dt.to_period('M')

dataset_start = delivered['order_purchase_timestamp'].min()
dataset_end = delivered['order_purchase_timestamp'].max()
dataset_end_month = delivered['order_month'].max()

print('Período entregue:', dataset_start, '→', dataset_end)
print('Último mês observado:', dataset_end_month)
print('Pedidos entregues:', f'{len(delivered):,}')
print('Clientes únicos:', f"{delivered['customer_unique_id'].nunique():,}")
print('Receita de pagamentos:', f"R$ {delivered['payment_value'].fillna(0).sum():,.2f}")

## 3. Definição das coortes

A coorte de cada cliente é o **mês de sua primeira compra entregue**.

O `cohort_index` mede quantos meses se passaram desde a aquisição:
- `M0`: mês da primeira compra;
- `M1`: mês seguinte;
- `M2`: dois meses depois;
- etc.

In [ ]:
first_purchase_month = (
    delivered.groupby('customer_unique_id')['order_month']
    .min()
    .rename('cohort_month')
)

delivered = delivered.merge(
    first_purchase_month,
    on='customer_unique_id',
    how='left',
    validate='many_to_one'
)

delivered['cohort_index'] = (
    (delivered['order_month'].dt.year - delivered['cohort_month'].dt.year) * 12 +
    (delivered['order_month'].dt.month - delivered['cohort_month'].dt.month)
).astype(int)

assert (delivered['cohort_index'] >= 0).all()
display(delivered[['customer_unique_id', 'order_id', 'order_month', 'cohort_month', 'cohort_index']].head())

## 4. Aquisição por coorte

Antes da retenção, avaliamos o tamanho das safras. Isso ajuda a separar mudanças de retenção de simples mudanças no volume de aquisição.

In [ ]:
cohort_sizes = (
    first_purchase_month.reset_index()
    .groupby('cohort_month')
    .agg(acquired_customers=('customer_unique_id', 'nunique'))
    .reset_index()
    .sort_values('cohort_month')
)
cohort_sizes['cohort_month_str'] = cohort_sizes['cohort_month'].astype(str)

display(cohort_sizes)

plt.figure(figsize=(12, 5))
plt.bar(cohort_sizes['cohort_month_str'], cohort_sizes['acquired_customers'])
plt.title('Clientes adquiridos por coorte mensal')
plt.xlabel('Mês da primeira compra')
plt.ylabel('Clientes adquiridos')
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

## 5. Matriz de retenção mensal de clientes

A retenção em `Mk` é a proporção de clientes da coorte que fizeram **pelo menos uma compra naquele mês específico**.

Para evitar **censura à direita**, células de meses que ainda não poderiam ter sido observados são exibidas como `NaN`, e não como zero. Assim, uma coorte recente não é penalizada por não possuir M6 ou M12 observável.

In [ ]:
cohort_customers = (
    delivered.groupby(['cohort_month', 'cohort_index'])['customer_unique_id']
    .nunique()
    .unstack(fill_value=0)
    .sort_index()
)

max_age = int(max(
    (dataset_end_month.year - cm.year) * 12 + (dataset_end_month.month - cm.month)
    for cm in cohort_customers.index
))
cohort_customers = cohort_customers.reindex(columns=range(max_age + 1), fill_value=0)

cohort_size_series = cohort_customers[0].copy()
customer_retention = cohort_customers.div(cohort_size_series, axis=0)

# Mascara meses não observáveis para cada coorte.
for cm in customer_retention.index:
    observed_age = (dataset_end_month.year - cm.year) * 12 + (dataset_end_month.month - cm.month)
    customer_retention.loc[cm, customer_retention.columns > observed_age] = np.nan

customer_retention.index = customer_retention.index.astype(str)
customer_retention.index.name = 'cohort_month'
customer_retention.columns = [f'M{i}' for i in customer_retention.columns]

display((customer_retention.iloc[:, :13] * 100).round(2))

In [ ]:
retention_plot = customer_retention.iloc[:, :13] * 100

plt.figure(figsize=(14, 8))
img = plt.imshow(retention_plot.values, aspect='auto')
plt.colorbar(img, label='Retenção (%)')
plt.title('Heatmap de retenção mensal por coorte — M0 a M12')
plt.xlabel('Idade da coorte')
plt.ylabel('Coorte de aquisição')
plt.xticks(range(len(retention_plot.columns)), retention_plot.columns)
plt.yticks(range(len(retention_plot.index)), retention_plot.index)
plt.tight_layout()
plt.show()

## 6. Curva agregada de retenção

Para cada idade da coorte, calculamos a retenção sobre **clientes elegíveis**, isto é, apenas clientes pertencentes a coortes antigas o suficiente para que aquele mês pudesse ser observado.

Essa leitura é mais segura do que preencher meses futuros com zero.

In [ ]:
customer_cohort = first_purchase_month.reset_index()
customer_cohort['cohort_age_at_end'] = (
    (dataset_end_month.year - customer_cohort['cohort_month'].dt.year) * 12 +
    (dataset_end_month.month - customer_cohort['cohort_month'].dt.month)
)

activity = delivered[['customer_unique_id', 'cohort_index']].drop_duplicates()

weighted_retention_rows = []
for month_index in range(0, min(max_age, 18) + 1):
    eligible_ids = customer_cohort.loc[
        customer_cohort['cohort_age_at_end'] >= month_index,
        'customer_unique_id'
    ]
    active_ids = activity.loc[activity['cohort_index'] == month_index, 'customer_unique_id']
    retained = eligible_ids.isin(set(active_ids)).sum()
    weighted_retention_rows.append({
        'cohort_index': month_index,
        'eligible_customers': len(eligible_ids),
        'active_customers': int(retained),
        'retention_rate': retained / len(eligible_ids) if len(eligible_ids) else np.nan
    })

weighted_retention = pd.DataFrame(weighted_retention_rows)
display(weighted_retention.head(13))

plt.figure(figsize=(10, 5))
plot_wr = weighted_retention.loc[weighted_retention['cohort_index'] <= 12]
plt.plot(plot_wr['cohort_index'], plot_wr['retention_rate'] * 100, marker='o')
plt.title('Retenção mensal agregada — clientes elegíveis')
plt.xlabel('Meses desde a aquisição')
plt.ylabel('Retenção (%)')
plt.xticks(plot_wr['cohort_index'])
plt.tight_layout()
plt.show()

### Retenção mensal não é taxa de recompra acumulada

Um cliente ativo em M1 pode não comprar em M2 e retornar em M3. Por isso, a curva acima responde **“qual percentual comprou exatamente nesse mês?”**.

Para uma leitura mais próxima de relacionamento, calcularemos agora **“qual percentual já realizou uma segunda compra até essa idade da coorte?”**.

## 7. Recompra acumulada por idade da coorte

A segunda compra é identificada pela sequência temporal de pedidos de cada cliente. O índice mensal da segunda compra pode ser `0` quando o cliente faz duas compras no próprio mês de aquisição.

Para que a curva acumulada tenha **o mesmo denominador em todos os pontos**, usamos apenas clientes com **12 meses completos de observação**. Assim, a taxa até M1, M3, M6 e M12 representa a evolução da mesma população madura e é naturalmente não decrescente.

> Essa curva é uma leitura longitudinal das coortes maduras. Para janelas mais recentes e com maior cobertura da base, o Notebook 04 traz as taxas ajustadas de 30/60/90/180 dias.


In [ ]:
ordered = delivered.sort_values(['customer_unique_id', 'order_purchase_timestamp', 'order_id']).copy()
ordered['order_number'] = ordered.groupby('customer_unique_id').cumcount() + 1

second_purchase = (
    ordered.loc[ordered['order_number'] == 2, ['customer_unique_id', 'order_month', 'cohort_month']]
    .copy()
)
second_purchase['second_purchase_index'] = (
    (second_purchase['order_month'].dt.year - second_purchase['cohort_month'].dt.year) * 12 +
    (second_purchase['order_month'].dt.month - second_purchase['cohort_month'].dt.month)
).astype(int)

customer_repeat_age = customer_cohort.merge(
    second_purchase[['customer_unique_id', 'second_purchase_index']],
    on='customer_unique_id', how='left', validate='one_to_one'
)

FOLLOWUP_MONTHS = 12
mature_repeat_base = customer_repeat_age.loc[
    customer_repeat_age['cohort_age_at_end'] >= FOLLOWUP_MONTHS
].copy()

cumulative_repeat_rows = []
for month_index in range(0, FOLLOWUP_MONTHS + 1):
    repeated = (
        mature_repeat_base['second_purchase_index'].notna() &
        (mature_repeat_base['second_purchase_index'] <= month_index)
    )
    cumulative_repeat_rows.append({
        'cohort_index': month_index,
        'eligible_customers': len(mature_repeat_base),
        'customers_with_second_purchase': int(repeated.sum()),
        'cumulative_repeat_rate': repeated.mean()
    })

cumulative_repeat = pd.DataFrame(cumulative_repeat_rows)
display(cumulative_repeat)

plt.figure(figsize=(10, 5))
plt.plot(cumulative_repeat['cohort_index'], cumulative_repeat['cumulative_repeat_rate'] * 100, marker='o')
plt.title('Recompra acumulada — clientes com 12 meses completos de observação')
plt.xlabel('Meses desde a aquisição')
plt.ylabel('Clientes com segunda compra acumulada (%)')
plt.xticks(cumulative_repeat['cohort_index'])
plt.tight_layout()
plt.show()


## 8. Evolução de M1, M2, M3 e M6 por safra

Essa visão permite verificar se a qualidade da aquisição melhorou ao longo do tempo. Para comparação executiva, descartamos coortes muito pequenas, que podem gerar taxas instáveis.

In [ ]:
ret_long = (
    customer_retention.reset_index()
    .melt(id_vars='cohort_month', var_name='month_label', value_name='retention_rate')
)
ret_long['cohort_index'] = ret_long['month_label'].str.replace('M', '', regex=False).astype(int)

cohort_comparison = cohort_sizes[['cohort_month', 'acquired_customers']].copy()
cohort_comparison['cohort_month'] = cohort_comparison['cohort_month'].astype(str)

for k in [1, 2, 3, 6, 12]:
    temp = ret_long.loc[ret_long['cohort_index'] == k, ['cohort_month', 'retention_rate']]
    temp = temp.rename(columns={'retention_rate': f'retention_M{k}'})
    cohort_comparison = cohort_comparison.merge(temp, on='cohort_month', how='left')

display(cohort_comparison)

In [ ]:
MIN_COHORT_SIZE = 500
mature_m1 = cohort_comparison.loc[
    (cohort_comparison['acquired_customers'] >= MIN_COHORT_SIZE) &
    cohort_comparison['retention_M1'].notna()
].copy()

plt.figure(figsize=(12, 5))
plt.plot(mature_m1['cohort_month'], mature_m1['retention_M1'] * 100, marker='o')
plt.title(f'Retenção M1 por coorte — mínimo {MIN_COHORT_SIZE} clientes')
plt.xlabel('Coorte de aquisição')
plt.ylabel('Retenção M1 (%)')
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

print('Top 5 coortes por M1:')
display(mature_m1.nlargest(5, 'retention_M1')[['cohort_month', 'acquired_customers', 'retention_M1']])
print('Bottom 5 coortes por M1:')
display(mature_m1.nsmallest(5, 'retention_M1')[['cohort_month', 'acquired_customers', 'retention_M1']])

## 9. Receita por coorte

Além de clientes ativos, medimos a receita de pagamentos associada a cada coorte em cada idade mensal.

`revenue_retention` abaixo é um **índice de receita relativa a M0**:

\[
	ext{Revenue Index}_{Mk} = rac{	ext{Receita da coorte em Mk}}{	ext{Receita da coorte em M0}}
\]

Ele não deve ser interpretado como NRR de SaaS, pois o Olist é um marketplace transacional sem receita contratual recorrente.

In [ ]:
cohort_revenue = (
    delivered.assign(payment_value=delivered['payment_value'].fillna(0))
    .groupby(['cohort_month', 'cohort_index'])['payment_value']
    .sum()
    .unstack(fill_value=0)
    .sort_index()
    .reindex(columns=range(max_age + 1), fill_value=0)
)

revenue_retention = cohort_revenue.div(cohort_revenue[0].replace(0, np.nan), axis=0)

for cm in revenue_retention.index:
    observed_age = (dataset_end_month.year - cm.year) * 12 + (dataset_end_month.month - cm.month)
    revenue_retention.loc[cm, revenue_retention.columns > observed_age] = np.nan

revenue_retention.index = revenue_retention.index.astype(str)
revenue_retention.index.name = 'cohort_month'
revenue_retention.columns = [f'M{i}' for i in revenue_retention.columns]

display((revenue_retention.iloc[:, :13] * 100).round(2))

In [ ]:
revenue_plot = revenue_retention.iloc[:, :13] * 100
plt.figure(figsize=(14, 8))
img = plt.imshow(revenue_plot.values, aspect='auto')
plt.colorbar(img, label='Receita vs M0 (%)')
plt.title('Heatmap do índice de receita por coorte — M0 a M12')
plt.xlabel('Idade da coorte')
plt.ylabel('Coorte de aquisição')
plt.xticks(range(len(revenue_plot.columns)), revenue_plot.columns)
plt.yticks(range(len(revenue_plot.index)), revenue_plot.index)
plt.tight_layout()
plt.show()

## 10. Receita acumulada por cliente adquirido — proxy de LTV observado

Uma coorte pode ter baixa retenção mensal e ainda assim acumular valor com recompras esparsas. Para acompanhar essa dinâmica, calculamos a receita acumulada da coorte dividida pelo número de clientes originalmente adquiridos.

Esse indicador é uma **proxy descritiva de LTV observado**, não um CLV predito.

In [ ]:
cohort_revenue_raw = (
    delivered.assign(payment_value=delivered['payment_value'].fillna(0))
    .groupby(['cohort_month', 'cohort_index'])['payment_value']
    .sum()
    .unstack(fill_value=0)
    .sort_index()
    .reindex(columns=range(max_age + 1), fill_value=0)
)

cohort_sizes_period = first_purchase_month.value_counts().sort_index()
cumulative_revenue_per_customer = cohort_revenue_raw.cumsum(axis=1).div(cohort_sizes_period, axis=0)

for cm in cumulative_revenue_per_customer.index:
    observed_age = (dataset_end_month.year - cm.year) * 12 + (dataset_end_month.month - cm.month)
    cumulative_revenue_per_customer.loc[cm, cumulative_revenue_per_customer.columns > observed_age] = np.nan

cumulative_revenue_per_customer.index = cumulative_revenue_per_customer.index.astype(str)
cumulative_revenue_per_customer.index.name = 'cohort_month'
cumulative_revenue_per_customer.columns = [f'M{i}' for i in cumulative_revenue_per_customer.columns]

display(cumulative_revenue_per_customer.iloc[:, :13].round(2))

In [ ]:
# Curvas para coortes com pelo menos 6 meses de observação e tamanho relevante.
eligible_curves = cohort_comparison.loc[
    (cohort_comparison['acquired_customers'] >= MIN_COHORT_SIZE) &
    cohort_comparison['retention_M6'].notna(),
    'cohort_month'
].tolist()

# Seleciona até 5 coortes espaçadas no tempo para evitar poluição visual.
if len(eligible_curves) > 5:
    idx = np.linspace(0, len(eligible_curves) - 1, 5).round().astype(int)
    selected_cohorts = [eligible_curves[i] for i in idx]
else:
    selected_cohorts = eligible_curves

plt.figure(figsize=(10, 6))
for cohort in selected_cohorts:
    row = cumulative_revenue_per_customer.loc[cohort, [f'M{i}' for i in range(0, 7)]]
    plt.plot(range(0, 7), row.values, marker='o', label=cohort)
plt.title('Receita acumulada por cliente adquirido — coortes selecionadas')
plt.xlabel('Meses desde a aquisição')
plt.ylabel('Receita acumulada por cliente (R$)')
plt.xticks(range(0, 7))
plt.legend(title='Coorte')
plt.tight_layout()
plt.show()

## 11. Meio de pagamento da primeira compra × retenção

Como um pedido pode utilizar mais de um meio de pagamento, definimos `primary_payment_type` como o tipo com maior valor dentro do primeiro pedido do cliente.

A análise abaixo mede a retenção mensal exata por tipo de pagamento da aquisição. As diferenças representam **associação**, não causalidade.

In [ ]:
payment_type_value = (
    payments.groupby(['order_id', 'payment_type'], as_index=False)['payment_value']
    .sum()
)
primary_idx = payment_type_value.groupby('order_id')['payment_value'].idxmax()
primary_payment = (
    payment_type_value.loc[primary_idx, ['order_id', 'payment_type']]
    .rename(columns={'payment_type': 'primary_payment_type'})
)

first_orders = ordered.loc[ordered['order_number'] == 1, [
    'customer_unique_id', 'order_id', 'cohort_month'
]].rename(columns={'order_id': 'first_order_id'})

customer_acquisition = (
    customer_cohort
    .merge(first_orders[['customer_unique_id', 'first_order_id']], on='customer_unique_id', how='left', validate='one_to_one')
    .merge(
        primary_payment.rename(columns={'order_id': 'first_order_id'}),
        on='first_order_id', how='left', validate='one_to_one'
    )
)
customer_acquisition['primary_payment_type'] = customer_acquisition['primary_payment_type'].fillna('unknown')

display(customer_acquisition['primary_payment_type'].value_counts(dropna=False).to_frame('customers'))

In [ ]:
activity_pairs = activity.copy()
retention_payment_rows = []

for payment_type, group in customer_acquisition.groupby('primary_payment_type'):
    for month_index in range(1, 7):
        eligible = group.loc[group['cohort_age_at_end'] >= month_index, 'customer_unique_id']
        active_ids = activity_pairs.loc[activity_pairs['cohort_index'] == month_index, 'customer_unique_id']
        retained = eligible.isin(set(active_ids)).sum()
        retention_payment_rows.append({
            'primary_payment_type': payment_type,
            'cohort_index': month_index,
            'eligible_customers': len(eligible),
            'active_customers': int(retained),
            'retention_rate': retained / len(eligible) if len(eligible) else np.nan
        })

retention_by_payment = pd.DataFrame(retention_payment_rows)
display(retention_by_payment)

In [ ]:
payment_sizes = customer_acquisition['primary_payment_type'].value_counts()
major_payment_types = payment_sizes.loc[payment_sizes >= 500].index.tolist()
plot_pay = retention_by_payment.loc[retention_by_payment['primary_payment_type'].isin(major_payment_types)]

plt.figure(figsize=(10, 6))
for payment_type, group in plot_pay.groupby('primary_payment_type'):
    plt.plot(group['cohort_index'], group['retention_rate'] * 100, marker='o', label=payment_type)
plt.title('Retenção mensal por meio de pagamento da primeira compra')
plt.xlabel('Meses desde a aquisição')
plt.ylabel('Retenção (%)')
plt.xticks(range(1, 7))
plt.legend(title='Pagamento principal')
plt.tight_layout()
plt.show()

## 12. Resumo executivo por coorte

Consolidamos tamanho, receita de aquisição e retenção em marcos relevantes. Essa tabela pode alimentar diretamente o relatório executivo ou um dashboard.

In [ ]:
acquisition_revenue = (
    delivered.loc[delivered['cohort_index'] == 0]
    .groupby('cohort_month')['payment_value']
    .sum()
    .rename('acquisition_month_revenue')
    .reset_index()
)

cohort_summary = cohort_sizes[['cohort_month', 'acquired_customers']].copy()
cohort_summary = cohort_summary.merge(acquisition_revenue, on='cohort_month', how='left')
cohort_summary['avg_acquisition_month_revenue_per_customer'] = (
    cohort_summary['acquisition_month_revenue'] / cohort_summary['acquired_customers']
)
cohort_summary['cohort_month'] = cohort_summary['cohort_month'].astype(str)

for k in [1, 2, 3, 6, 12]:
    temp = ret_long.loc[ret_long['cohort_index'] == k, ['cohort_month', 'retention_rate']]
    temp = temp.rename(columns={'retention_rate': f'retention_M{k}'})
    cohort_summary = cohort_summary.merge(temp, on='cohort_month', how='left')

for k in [3, 6, 12]:
    col = f'M{k}'
    if col in cumulative_revenue_per_customer.columns:
        temp = cumulative_revenue_per_customer[[col]].reset_index().rename(columns={col: f'cum_revenue_per_customer_M{k}'})
        cohort_summary = cohort_summary.merge(temp, on='cohort_month', how='left')

display(cohort_summary)

## 13. KPIs executivos

Para retenção em `Mk`, o denominador inclui apenas clientes de coortes com `Mk` completamente observável.

In [ ]:
def weighted_retention_at(k):
    row = weighted_retention.loc[weighted_retention['cohort_index'] == k]
    return row['retention_rate'].iloc[0] if len(row) else np.nan

def cumulative_repeat_at(k):
    row = cumulative_repeat.loc[cumulative_repeat['cohort_index'] == k]
    return row['cumulative_repeat_rate'].iloc[0] if len(row) else np.nan

repeat_customers = int(second_purchase['customer_unique_id'].nunique())
raw_repeat_rate = repeat_customers / delivered['customer_unique_id'].nunique()

executive_kpis = pd.DataFrame({
    'kpi': [
        'customers_delivered',
        'cohorts_observed',
        'raw_repeat_rate_any_time',
        'retention_M1_weighted',
        'retention_M2_weighted',
        'retention_M3_weighted',
        'retention_M6_weighted',
        'retention_M12_weighted',
        'cumulative_repeat_by_M1',
        'cumulative_repeat_by_M3',
        'cumulative_repeat_by_M6',
        'cumulative_repeat_by_M12'
    ],
    'value': [
        delivered['customer_unique_id'].nunique(),
        cohort_sizes['cohort_month'].nunique(),
        raw_repeat_rate,
        weighted_retention_at(1),
        weighted_retention_at(2),
        weighted_retention_at(3),
        weighted_retention_at(6),
        weighted_retention_at(12),
        cumulative_repeat_at(1),
        cumulative_repeat_at(3),
        cumulative_repeat_at(6),
        cumulative_repeat_at(12)
    ]
})
display(executive_kpis)

In [ ]:
print('LEITURA EXECUTIVA')
print('-' * 90)
print(f"1. A base entregue reúne {delivered['customer_unique_id'].nunique():,} clientes distribuídos em {cohort_sizes['cohort_month'].nunique()} coortes mensais.")
print(f"2. A retenção mensal ponderada é {weighted_retention_at(1):.2%} em M1, {weighted_retention_at(3):.2%} em M3 e {weighted_retention_at(6):.2%} em M6.")
print(f"3. Entre clientes com 12 meses completos de observação, a recompra acumulada atinge {cumulative_repeat_at(1):.2%} até M1, {cumulative_repeat_at(3):.2%} até M3 e {cumulative_repeat_at(6):.2%} até M6.")
print(f"4. A taxa bruta de clientes com pelo menos dois pedidos entregues no período completo é {raw_repeat_rate:.2%}.")
print('5. Retenção exata e recompra acumulada respondem perguntas diferentes e devem aparecer separadamente no storytelling.')
print('6. Coortes recentes devem ser comparadas somente em idades já observáveis; células futuras não podem ser tratadas como zero.')
print('7. Receita acumulada por cliente ajuda a avaliar a qualidade econômica das safras, mesmo quando a frequência mensal é baixa.')


## 14. Recomendações para storytelling executivo

### Tese 1 — Crescimento de aquisição não garante crescimento de relacionamento
O número de clientes adquiridos aumenta fortemente ao longo da série, mas a retenção mensal permanece baixa. Isso sugere que crescimento de volume pode depender mais de aquisição do que de recorrência.

### Tese 2 — A leitura de retenção precisa respeitar maturidade das coortes
Comparar uma coorte antiga em M6 com uma coorte recente sem seis meses de observação gera viés. O heatmap mascarado e a curva de clientes elegíveis corrigem essa distorção.

### Tese 3 — Retenção mensal e recompra acumulada são complementares
A retenção mensal mostra a frequência de atividade em cada mês; a recompra acumulada mostra quantos clientes já foram convertidos para uma segunda compra. Em um marketplace, a segunda métrica costuma ser mais intuitiva para decisões de CRM.

### Tese 4 — Valor econômico deve acompanhar retenção
A receita acumulada por cliente adquirido funciona como uma proxy de LTV observado e ajuda a distinguir coortes que voltam pouco, mas retornam com tickets relevantes, de coortes com baixo valor incremental.

### Tese 5 — Pagamento é um sinal de segmentação, não uma causa
Diferenças de retenção por meio de pagamento podem refletir categoria, ticket, período e perfil do cliente. O meio de pagamento deve entrar como feature do modelo preditivo, e não como explicação causal isolada.

### Ação sugerida
Concentrar esforços de CRM na conversão da primeira para a segunda compra e acompanhar cada nova safra por **M1/M3 + recompra acumulada + receita acumulada por cliente**. Isso cria um scorecard de aquisição muito mais informativo do que olhar somente pedidos ou GMV total.

## 15. Exportação das camadas analíticas

In [ ]:
# Versões numéricas com nomes estáveis para uso em BI/modelagem.
customer_retention_export = customer_retention.reset_index()
revenue_retention_export = revenue_retention.reset_index()
cumulative_revenue_export = cumulative_revenue_per_customer.reset_index()

exports = {
    'cohort_customer_retention_matrix.csv': customer_retention_export,
    'cohort_weighted_retention_curve.csv': weighted_retention,
    'cohort_cumulative_repeat_curve.csv': cumulative_repeat,
    'cohort_revenue_index_matrix.csv': revenue_retention_export,
    'cohort_cumulative_revenue_per_customer.csv': cumulative_revenue_export,
    'cohort_retention_by_first_payment.csv': retention_by_payment,
    'cohort_summary.csv': cohort_summary,
    'cohort_executive_kpis.csv': executive_kpis
}

for filename, df in exports.items():
    path = PROCESSED_DIR / filename
    df.to_csv(path, index=False)
    print('Exportado:', path, df.shape)

## 16. Sanity checks

In [ ]:
assert delivered['cohort_index'].ge(0).all(), 'Há cohort_index negativo.'
assert np.allclose(customer_retention['M0'].dropna(), 1), 'M0 da retenção deve ser 100%.'
assert customer_retention.drop(columns=['M0']).stack().between(0, 1).all(), 'Retenção fora de [0,1].'
assert weighted_retention['retention_rate'].between(0, 1).all(), 'Curva agregada fora de [0,1].'
assert cumulative_repeat['cumulative_repeat_rate'].between(0, 1).all(), 'Recompra acumulada fora de [0,1].'
assert cumulative_repeat['cumulative_repeat_rate'].is_monotonic_increasing, 'Com denominador fixo, a recompra acumulada deve ser não decrescente.'
assert (second_purchase['second_purchase_index'] >= 0).all(), 'Há segunda compra anterior à coorte de aquisição.'
assert customer_acquisition['customer_unique_id'].is_unique, 'Base de aquisição deve ter uma linha por cliente.'
assert cohort_summary['cohort_month'].is_unique, 'Resumo deve ter uma linha por coorte.'

print('Sanity checks concluídos com sucesso.')


## Conclusão

O Notebook 05 transforma a base de pedidos em uma visão longitudinal de aquisição, retenção e valor por safra.

As principais saídas para decisão são:
1. **Heatmap de retenção mensal** com correção de censura à direita;
2. **Curva de recompra acumulada** para acompanhar a conversão à segunda compra;
3. **Índice de receita por coorte** e **receita acumulada por cliente** como proxy descritiva de LTV;
4. **Retenção por meio de pagamento da primeira compra** como sinal de segmentação;
5. **Scorecard por coorte** com M1, M3, M6 e valor acumulado.

O próximo passo recomendado é o **Notebook 06 — Repurchase Prediction**, usando a base preparada no Notebook 04 para estimar a probabilidade de recompra em 90 dias com validação temporal e tratamento do desbalanceamento do target.